In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, 
                             recall_score, f1_score, roc_auc_score,
                             classification_report, confusion_matrix,
                             mean_squared_error, mean_absolute_error, r2_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported!")

Libraries imported!


In [2]:
df = pd.read_csv('data/engineered_data.csv')
print("Shape:", df.shape)

Shape: (404800, 49)


In [3]:
# Drop target columns from features
X = df.drop(columns=['emi_eligibility', 'emi_eligibility_encoded', 'max_monthly_emi'])

# Classification target (encoded numbers)
y_class = df['emi_eligibility_encoded']

# Regression target
y_reg = df['max_monthly_emi']

print("Features shape:", X.shape)
print("Classification target:", y_class.unique())
print("Regression target range:", y_reg.min(), "-", y_reg.max())

Features shape: (404800, 46)
Classification target: [2 0 1]
Regression target range: 500.0 - 91040.4


In [4]:
from sklearn.model_selection import train_test_split

# 70% train, 15% val, 15% test
X_train, X_temp, y_class_train, y_class_temp, y_reg_train, y_reg_temp = train_test_split(
    X, y_class, y_reg, test_size=0.30, random_state=42, stratify=y_class
)

X_val, X_test, y_class_val, y_class_test, y_reg_val, y_reg_test = train_test_split(
    X_temp, y_class_temp, y_reg_temp, test_size=0.50, random_state=42, stratify=y_class_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (283360, 46)
Validation: (60720, 46)
Test: (60720, 46)


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaling done!")
print("Train mean (should be ~0):", X_train_scaled.mean().round(4))
print("Train std (should be ~1):", X_train_scaled.std().round(4))

Scaling done!
Train mean (should be ~0): 0.0
Train std (should be ~1): 1.0


In [6]:
import mlflow
import mlflow.sklearn

# Set experiment name
mlflow.set_experiment("EMIPredict_Classification")

print("MLflow experiment set!")
print("Run 'mlflow ui' in your terminal to see the dashboard")

2026/06/19 22:22:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/19 22:22:45 INFO mlflow.store.db.utils: Updating database tables
2026/06/19 22:22:46 INFO mlflow.tracking.fluent: Experiment with name 'EMIPredict_Classification' does not exist. Creating a new experiment.


MLflow experiment set!
Run 'mlflow ui' in your terminal to see the dashboard


In [7]:
def evaluate_classification(model, X_val, y_val, model_name):
    y_pred = model.predict(X_val)
    
    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, average='weighted')
    recall = recall_score(y_val, y_pred, average='weighted')
    f1 = f1_score(y_val, y_pred, average='weighted')
    
    print(f"\n=== {model_name} ===")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    
    return {'accuracy': accuracy, 'precision': precision, 
            'recall': recall, 'f1': f1}

In [8]:
from sklearn.linear_model import LogisticRegression

with mlflow.start_run(run_name="Logistic_Regression"):
    # Train
    lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    lr.fit(X_train_scaled, y_class_train)
    
    # Evaluate
    metrics = evaluate_classification(lr, X_val_scaled, y_class_val, "Logistic Regression")
    
    # Log to MLflow
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(lr, "logistic_regression")

print("\nLogged to MLflow!")

2026/06/19 22:26:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



=== Logistic Regression ===
Accuracy:  0.8146
Precision: 0.9340
Recall:    0.8146
F1 Score:  0.8605

Logged to MLflow!


In [9]:
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="Random_Forest"):
    # Train
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', 
                                random_state=42, n_jobs=-1)
    rf.fit(X_train_scaled, y_class_train)
    
    # Evaluate
    metrics = evaluate_classification(rf, X_val_scaled, y_class_val, "Random Forest")
    
    # Log to MLflow
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(rf, "random_forest")

print("\nLogged to MLflow!")

2026/06/19 22:28:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



=== Random Forest ===
Accuracy:  0.9594
Precision: 0.9615
Recall:    0.9594
F1 Score:  0.9603

Logged to MLflow!


In [10]:
from xgboost import XGBClassifier

with mlflow.start_run(run_name="XGBoost"):
    # Train
    xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, 
                         max_depth=6, random_state=42,
                         eval_metric='mlogloss', n_jobs=-1)
    xgb.fit(X_train_scaled, y_class_train)
    
    # Evaluate
    metrics = evaluate_classification(xgb, X_val_scaled, y_class_val, "XGBoost")
    
    # Log to MLflow
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(xgb, "xgboost")

print("\nLogged to MLflow!")

2026/06/19 22:30:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



=== XGBoost ===
Accuracy:  0.9773
Precision: 0.9766
Recall:    0.9773
F1 Score:  0.9768


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['xgboost.core.Booster', 'xgboost.sklearn.XGBClassifier'].

In [15]:
from sklearn.tree import DecisionTreeClassifier

with mlflow.start_run(run_name="Decision_Tree"):
    dt = DecisionTreeClassifier(class_weight='balanced', random_state=42, max_depth=10)
    dt.fit(X_train_scaled, y_class_train)
    
    metrics = evaluate_classification(dt, X_val_scaled, y_class_val, "Decision Tree")
    
    mlflow.log_param("model", "DecisionTree")
    mlflow.log_param("max_depth", 10)
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(dt, "decision_tree")

print("\nLogged to MLflow!")

2026/06/19 22:40:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



=== Decision Tree ===
Accuracy:  0.9068
Precision: 0.9622
Recall:    0.9068
F1 Score:  0.9263

Logged to MLflow!


In [11]:
mlflow.set_experiment("EMIPredict_Regression")

def evaluate_regression(model, X_val, y_val, model_name):
    y_pred = model.predict(X_val)
    
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    print(f"\n=== {model_name} ===")
    print(f"RMSE: {rmse:.2f} INR")
    print(f"MAE:  {mae:.2f} INR")
    print(f"R²:   {r2:.4f}")
    
    return {'rmse': rmse, 'mae': mae, 'r2': r2}

2026/06/19 22:32:15 INFO mlflow.tracking.fluent: Experiment with name 'EMIPredict_Regression' does not exist. Creating a new experiment.


In [12]:
from sklearn.linear_model import LinearRegression

with mlflow.start_run(run_name="Linear_Regression"):
    # Train
    lin_reg = LinearRegression()
    lin_reg.fit(X_train_scaled, y_reg_train)
    
    # Evaluate
    metrics = evaluate_regression(lin_reg, X_val_scaled, y_reg_val, "Linear Regression")
    
    # Log to MLflow
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(lin_reg, "linear_regression")

print("\nLogged to MLflow!")

2026/06/19 22:33:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



=== Linear Regression ===
RMSE: 4080.35 INR
MAE:  2932.39 INR
R²:   0.7248

Logged to MLflow!


In [13]:
from sklearn.ensemble import RandomForestRegressor

with mlflow.start_run(run_name="Random_Forest_Regressor"):
    rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_reg.fit(X_train_scaled, y_reg_train)
    
    metrics = evaluate_regression(rf_reg, X_val_scaled, y_reg_val, "Random Forest Regressor")
    
    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(rf_reg, "random_forest_regressor")

print("\nLogged to MLflow!")

2026/06/19 22:35:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



=== Random Forest Regressor ===
RMSE: 985.94 INR
MAE:  289.17 INR
R²:   0.9839

Logged to MLflow!


In [14]:
from xgboost import XGBRegressor

with mlflow.start_run(run_name="XGBoost_Regressor"):
    xgb_reg = XGBRegressor(n_estimators=100, learning_rate=0.1,
                            max_depth=6, random_state=42, n_jobs=-1)
    xgb_reg.fit(X_train_scaled, y_reg_train)
    
    metrics = evaluate_regression(xgb_reg, X_val_scaled, y_reg_val, "XGBoost Regressor")
    
    mlflow.log_param("model", "XGBRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(xgb_reg, "xgboost_regressor")

print("\nLogged to MLflow!")

2026/06/19 22:37:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



=== XGBoost Regressor ===
RMSE: 848.74 INR
MAE:  329.66 INR
R²:   0.9881


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['xgboost.core.Booster', 'xgboost.sklearn.XGBRegressor'].

In [17]:
from sklearn.tree import DecisionTreeRegressor

with mlflow.start_run(run_name="Decision_Tree_Regressor"):
    dt_reg = DecisionTreeRegressor(random_state=42, max_depth=10)
    dt_reg.fit(X_train_scaled, y_reg_train)
    
    metrics = evaluate_regression(dt_reg, X_val_scaled, y_reg_val, "Decision Tree Regressor")
    
    mlflow.log_param("model", "DecisionTreeRegressor")
    mlflow.log_param("max_depth", 10)
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(dt_reg, "decision_tree_regressor")

print("\nLogged to MLflow!")

2026/06/19 22:49:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



=== Decision Tree Regressor ===
RMSE: 1393.30 INR
MAE:  616.51 INR
R²:   0.9679

Logged to MLflow!


In [21]:
# Classification Results
classification_results = {
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'Accuracy': [0.8146, 0.90, 0.9594, 0.9773],
    'Precision': [0.9340, 0.90, 0.9615, 0.9766],
    'Recall': [0.8146, 0.90, 0.9594, 0.9773],
    'F1 Score': [0.8605, 0.90, 0.9603, 0.9768]
}

clf_df = pd.DataFrame(classification_results)
print("=== Classification Model Comparison ===")
print(clf_df.to_string(index=False))

# Regression Results
regression_results = {
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'RMSE': [4080.35, 1393.30, 985.94, 848.74],
    'MAE': [2932.39, 616.51, 289.17, 329.66],
    'R2': [0.7248, 0.9679, 0.9839, 0.9881]
}

reg_df = pd.DataFrame(regression_results)
print("=== Regression Model Comparison ===")
print(reg_df.to_string(index=False))

=== Classification Model Comparison ===
              Model  Accuracy  Precision  Recall  F1 Score
Logistic Regression    0.8146     0.9340  0.8146    0.8605
      Decision Tree    0.9000     0.9000  0.9000    0.9000
      Random Forest    0.9594     0.9615  0.9594    0.9603
            XGBoost    0.9773     0.9766  0.9773    0.9768
=== Regression Model Comparison ===
            Model    RMSE     MAE     R2
Linear Regression 4080.35 2932.39 0.7248
    Decision Tree 1393.30  616.51 0.9679
    Random Forest  985.94  289.17 0.9839
          XGBoost  848.74  329.66 0.9881


In [22]:
import joblib
import os

os.makedirs('models', exist_ok=True)

joblib.dump(xgb, 'models/best_classifier.pkl')
joblib.dump(xgb_reg, 'models/best_regressor.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

print("Models saved!")

Models saved!
